<a href="https://colab.research.google.com/github/Daniuss/case-monks-operations/blob/main/Monk_Case.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

Saving opps_corrupted.xlsx to opps_corrupted.xlsx


In [25]:
df = pd.read_excel('opps_corrupted.xlsx')
print(f'Linhas: {len(df)}')
print(f'Colunas: {len(df.columns)}')
print(f'Oportunidades Únicas: {df["Opportunity_ID"].nunique()}')
df.head(3)

Linhas: 413
Colunas: 18
Oportunidades Únicas: 261


,Opportunity_ID,Account_ID,Account_Name,Opportunity_Owner,Opportunity_Name,Type,Stage,Amount,Amount_Currency,Total_Product_Amount,Total_Product_Amount_Currency,Product_Name,Close_Date,Created_Date,Lead_Source,Lead_Office,Stage_Duration,Delivery_Length_Months
0,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - DDM - Media - Managed Media (AO...,Change Order/Upsell,Registration,"3043949,95",BRL,"829077,09",BRL,DDM - Media - Managed Media (AOR),30/04/2026,19/05/2025,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12
1,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - Content - Social - AOR - 04/2026,Change Order/Upsell,Registration,"3043949,95",BRL,"1475924,6",BRL,Content - Social - AOR,30/04/2026,19/05/2025,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12
2,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - DDM - Media - Creative - 04/2026,Change Order/Upsell,Registration,"3043949,95",BRL,"738948,26",BRL,DDM - Media - Creative,30/04/2026,19/05/2025,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12


In [26]:
for col in ['Stage', 'Lead_Source', 'Lead_Office', 'Type']:
  print(f'\n=== {col} ===')
  print(df[col].value_counts(dropna=False).to_string())


=== Stage ===
Stage
Closed Won                152
Pitched                    67
Registration               63
Negotiation                51
Opportunity Identified     28
Pitching                   28
Qualified                  12
Clossed Won                 3
Pithced                     3
Pitchinng                   2
Closed Wonn                 1
Qualifyed                   1
Negociation                 1
Registrationn               1

=== Lead_Source ===
Lead_Source
Inbound - Current Client (DEPRECATED)             227
Referral - Internal                                28
Inbound - Website - Media.Monks (DEPRECATED)       25
Referral - Personal                                25
MH Lead (DEPRECATED)                               19
Referral - S4 Company                              16
Existing Client - Account/Growth Activity          16
Don't Know/Other (DEPRECATED)                       9
Prospecting - Personal (DEPRECATED)                 7
inbound - current client (deprecated)   

In [27]:
stage_corrections = {
    'Clossed Won': 'Closed Won',
    'Closed Wonn': 'Closed Won',
    'Pitchced': 'Pitching',
    'Pitchinng': 'Pitching',
    'Qualifyed': 'Qualified',
    'Negociation': 'Negaotiation',
    'Registrationn': 'Registration',
}

print(f'{df["Stage"].isin(stage_corrections.keys()).sum()} linhas corrigidas')
df['Stage'] = df['Stage'].replace(stage_corrections)

9 linhas corrigidas


In [28]:
ls_corrections = {
    'inbound - current client (deprecated)':        'Inbound - Current Client (DEPRECATED)',
    ' Inbound - Current Client (DEPRECATED)':       'Inbound - Current Client (DEPRECATED)',
    'Refferal - Internal':                          'Referral - Internal',
    'REFERRAL - INTERNAL':                          'Referral - Internal',
    ' Referral - Personal':                         'Referral - Personal',
    'inbound - website - media.monks (DEPRECATED)': 'Inbound - Website - Media.Monks (DEPRECATED)',
    "Don't Know-Other":                             "Don't Know/Other (DEPRECATED)",
    'existing client - account/growth activity':    'Existing Client - Account/Growth Activity',
    'Existing Client-Account/Growth Activity':      'Existing Client - Account/Growth Activity',
}

print(f'{df["Lead_Source"].isin(ls_corrections.keys()).sum()} linhas corrigidas')
df['Lead_Source'] = df['Lead_Source'].replace(ls_corrections)

15 linhas corrigidas


In [29]:
lo_corrections = {
    'São Paulo': 'Sao Paulo, BR',
    'sao paulo': 'Sao Paulo, BR',
    'SP':        'Sao Paulo, BR',
    'SAO PAULO': 'Sao Paulo, BR',
    'sp':        'Sao Paulo, BR',
}

print(f'{df["Lead_Office"].isin(lo_corrections.keys()).sum()} linhas corrigidas')
df['Lead_Office'] = df['Lead_Office'].replace(lo_corrections)

18 linhas corrigidas


In [30]:
def parse_br_number(val):
  if pd.isna(val):
    return np.nan
  s = str(val).strip().replace('R$', '').strip()
  if '.' in s and ',' in s:
    s = s.replace('.', '').replace(',', '.')
  elif ',' in s:
    parts = s.split(',')
    if len(parts) == 2 and len(parts[1]) == 3 and '.' not in parts[0]:
      s = s.replace(',', '')
    else:
      s = s.replace(',', '.')
  try:
    return float(s)
  except:
    return np.nan

df['Amount_Parsed'] = df['Amount'].apply(parse_br_number)
df['TPA_Parsed'] = df['Total_Product_Amount'].apply(parse_br_number)
print(f'Amount convertido com sucesso: {df["Amount_Parsed"].notna().sum()} valores')


Amount convertido com sucesso: 389 valores


In [31]:
opp_tpa_sum = df.groupby('Opportunity_ID')['TPA_Parsed'].sum()

divergent = 0
for opp_id, tpa_sum in opp_tpa_sum.items():
    amount_val = df.loc[df['Opportunity_ID'] == opp_id, 'Amount_Parsed'].iloc[0]
    if pd.notna(amount_val) and pd.notna(tpa_sum) and tpa_sum > 0:
        if abs(amount_val - tpa_sum) > 1:
            divergent += 1
            print(f'  {opp_id}: {amount_val:,.0f} >> {tpa_sum:,.0f}')
            df.loc[df['Opportunity_ID'] == opp_id, 'Amount_Parsed'] = tpa_sum

print(f'\n{divergent} oportunidades corrigidas')

  OPP-10027: 45,000 >> 36,000
  OPP-10030: 290,400 >> 645,420
  OPP-10081: 150,000 >> 120,000
  OPP-10089: 728,000 >> 520,000
  OPP-10092: 295,200 >> 492,000
  OPP-10097: 1,974,000 >> 1,410,000
  OPP-10111: 67,500 >> 90,000
  OPP-10126: 375,000 >> 300,000
  OPP-10151: 1,000 >> 420
  OPP-10196: 33,200 >> 44,271
  OPP-10220: 330,000 >> 264,000
  OPP-10259: 222,000 >> 370,000

12 oportunidades corrigidas


In [32]:
df['Delivery_Length_Months'] = df['Delivery_Length_Months'].apply(
    lambda x: str(x).replace(',', '.') if pd.notna(x) else x
)

lsc_map = {
    'Inbound - Website - Media.Monks (DEPRECATED)': 'Inbound',
    'Inbound - Marketing (DEPRECATED)':              'Inbound',
    'Inbound - Website (DEPRECATED)':               'Inbound',
    'Marketing - Lead Scoring/Nurturing':            'Inbound',
    'Website - Sales Form':                          'Inbound',
    'Prospecting - Personal (DEPRECATED)':           'Outbound',
    'Referral - Internal':                           'Referral',
    'Referral - Personal':                           'Referral',
    'Referral - S4 Company':                         'Referral',
    'Referral - Partner - Google Marketing Platform': 'Referral',
    'Referral - Partner - Google Cloud':              'Referral',
    'Inbound - Current Client (DEPRECATED)':         'Customer Success',
    'Existing Client - Account/Growth Activity':     'Customer Success',
    'Prospecting - Current Client (DEPRECATED)':     'Customer Success',
    'Industry Event':                                'Event',
    'MH Lead (DEPRECATED)':                          'Unknown',
    "Don't Know/Other (DEPRECATED)":                 'Unknown',
}

df['Lead_Source_Category'] = df['Lead_Source'].map(lsc_map)
print(df['Lead_Source_Category'].value_counts().to_string())

Lead_Source_Category
Customer Success    258
Referral             76
Inbound              33
Unknown              30
Outbound              7
Event                 4


In [33]:
def to_br_format(val):
    if pd.isna(val):
      return np.nan
    return f'{val:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

df['Amount'] = df['Amount_Parsed'].apply(to_br_format)
df['Total_Product_Amount'] = df['TPA_Parsed'].apply(to_br_format)
df.loc[df['Amount_Parsed'].isna(), 'Amount_Currency'] = np.nan
df.loc[df['Amount_Parsed'].notna() & df['Amount_Currency'].isna(), 'Amount_Currency'] = 'BRL'

output_cols = [
    'Opportunity_ID', 'Account_ID', 'Account_Name', 'Opportunity_Owner',
    'Opportunity_Name', 'Type', 'Stage', 'Amount', 'Amount_Currency',
    'Total_Product_Amount', 'Total_Product_Amount_Currency', 'Product_Name',
    'Close_Date', 'Created_Date', 'Lead_Source', 'Lead_Office',
    'Stage_Duration', 'Delivery_Length_Months', 'Lead_Source_Category'
]

df[output_cols].to_excel('opps_corrigido.xlsx', index=False)
files.download('opps_corrigido.xlsx')
df_out = df[output_cols]
print(f'opps_corrigido.xlsx gerado com sucesso!')
print(f'Linhas: {len(df_out)}')
print(f'Oportunidades únicas: {df_out["Opportunity_ID"].nunique()}')
print(f'Colunas: {len(df_out.columns)}')
print(f'\nColunas exportadas: {", ".join(output_cols)}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

opps_corrigido.xlsx gerado com sucesso!
Linhas: 413
Oportunidades únicas: 261
Colunas: 19

Colunas exportadas: Opportunity_ID, Account_ID, Account_Name, Opportunity_Owner, Opportunity_Name, Type, Stage, Amount, Amount_Currency, Total_Product_Amount, Total_Product_Amount_Currency, Product_Name, Close_Date, Created_Date, Lead_Source, Lead_Office, Stage_Duration, Delivery_Length_Months, Lead_Source_Category


Etapa 2. Relatório de Erros (relatorio_erros.html)

Nas próximas duas etapas vou percorrer a planilha original, comparar com as correções que fizemos, registrar cada problema encontrado e gerar um relatório HTML.

In [34]:
# Registro de todos os erros encontrados

import pandas as pd

df_original = pd.read_excel('opps_corrupted.xlsx')

errors_log = []

stage_corrections = {
    'Clossed Won': 'Closed Won',
    'Closed Wonn': 'Closed Won',
    'Pithced': 'Pitched',
    'Pitchinng': 'Pitching',
    'Qualifyed': 'Qualified',
    'Negociation': 'Negotiation',
    'Registrationn': 'Registration',
}
for idx, row in df_original.iterrows():
    if row['Stage'] in stage_corrections:
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Stage',
            'Categoria': 'Erro de grafia',
            'Valor original': row['Stage'],
            'Valor corrigido': stage_corrections[row['Stage']]
        })

ls_corrections = {
    'inbound - current client (deprecated)': 'Inbound - Current Client (DEPRECATED)',
    ' Inbound - Current Client (DEPRECATED)': 'Inbound - Current Client (DEPRECATED)',
    'Refferal - Internal': 'Referral - Internal',
    'REFERRAL - INTERNAL': 'Referral - Internal',
    ' Referral - Personal': 'Referral - Personal',
    'inbound - website - media.monks (DEPRECATED)': 'Inbound - Website - Media.Monks (DEPRECATED)',
    "Don't Know-Other": "Don't Know/Other (DEPRECATED)",
    'existing client - account/growth activity': 'Existing Client - Account/Growth Activity',
    'Existing Client-Account/Growth Activity': 'Existing Client - Account/Growth Activity',
}
for idx, row in df_original.iterrows():
    if row['Lead_Source'] in ls_corrections:
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Lead_Source',
            'Categoria': 'Erro de grafia',
            'Valor original': row['Lead_Source'],
            'Valor corrigido': ls_corrections[row['Lead_Source']]
        })

lo_corrections = {
    'São Paulo': 'Sao Paulo, BR',
    'sao paulo': 'Sao Paulo, BR',
    'SP': 'Sao Paulo, BR',
    'SAO PAULO': 'Sao Paulo, BR',
    'sp': 'Sao Paulo, BR',
}
for idx, row in df_original.iterrows():
    if row['Lead_Office'] in lo_corrections:
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Lead_Office',
            'Categoria': 'Erro de grafia',
            'Valor original': row['Lead_Office'],
            'Valor corrigido': lo_corrections[row['Lead_Office']]
        })

for idx, row in df_original.iterrows():
    val = row['Amount']
    if pd.notna(val):
        s = str(val).strip()
        if s.startswith('R$'):
            errors_log.append({
                'Linha': idx + 2,
                'Oportunidade': row['Opportunity_ID'],
                'Campo': 'Amount',
                'Categoria': 'Formato numérico',
                'Valor original': val,
                'Valor corrigido': s.replace('R$', '').strip()
            })
        elif s != str(val):
            errors_log.append({
                'Linha': idx + 2,
                'Oportunidade': row['Opportunity_ID'],
                'Campo': 'Amount',
                'Categoria': 'Formato numérico',
                'Valor original': repr(val),
                'Valor corrigido': s
            })

def parse_br(val):
    if pd.isna(val):
        return None
    s = str(val).strip().replace('R$', '').strip()
    if '.' in s and ',' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        parts = s.split(',')
        if len(parts) == 2 and len(parts[1]) == 3 and '.' not in parts[0]:
            s = s.replace(',', '')
        else:
            s = s.replace(',', '.')
    try:
        return float(s)
    except:
        return None

df_original['_amt'] = df_original['Amount'].apply(parse_br)
df_original['_tpa'] = df_original['Total_Product_Amount'].apply(parse_br)
opp_tpa = df_original.groupby('Opportunity_ID')['_tpa'].sum()

opps_divergentes = []
for opp_id, tpa_sum in opp_tpa.items():
    amt = df_original.loc[df_original['Opportunity_ID'] == opp_id, '_amt'].iloc[0]
    if pd.notna(amt) and pd.notna(tpa_sum) and tpa_sum > 0:
        if abs(amt - tpa_sum) > 1:
            opps_divergentes.append(opp_id)
            for idx in df_original[df_original['Opportunity_ID'] == opp_id].index:
                errors_log.append({
                    'Linha': idx + 2,
                    'Oportunidade': opp_id,
                    'Campo': 'Amount',
                    'Categoria': 'Divergência numérica',
                    'Valor original': f'{amt:,.2f}',
                    'Valor corrigido': f'{tpa_sum:,.2f}'
                })

valid_types = ['Project - Competitive', 'Project - Not Competitive', 'Change Order/Upsell']
for idx, row in df_original.iterrows():
    if pd.notna(row['Type']) and row['Type'] not in valid_types:
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Type',
            'Categoria': 'Fora do escopo',
            'Valor original': row['Type'],
            'Valor corrigido': '(excluído da análise)'
        })
    elif pd.isna(row['Type']):
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Type',
            'Categoria': 'Fora do escopo',
            'Valor original': 'vazio',
            'Valor corrigido': '(excluído da análise)'
        })

for idx, row in df_original.iterrows():
    if row.get('Amount_Currency') == 'USD':
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Amount_Currency',
            'Categoria': 'Inconsistência de moeda',
            'Valor original': 'USD',
            'Valor corrigido': '(sinalizado para revisão)'
        })

for idx, row in df_original.iterrows():
    val = row['Delivery_Length_Months']
    if pd.notna(val) and ',' in str(val):
        errors_log.append({
            'Linha': idx + 2,
            'Oportunidade': row['Opportunity_ID'],
            'Campo': 'Delivery_Length_Months',
            'Categoria': 'Formato numérico',
            'Valor original': val,
            'Valor corrigido': str(val).replace(',', '.')
        })

errors_df = pd.DataFrame(errors_log)
print(f'Total de problemas: {len(errors_log)}')
print(f'\nPor categoria:')
print(errors_df['Categoria'].value_counts().to_string())
print(f'\nPor campo:')
print(errors_df['Campo'].value_counts().to_string())

Total de problemas: 175

Por categoria:
Categoria
Fora do escopo             108
Erro de grafia              45
Divergência numérica        16
Formato numérico             5
Inconsistência de moeda      1

Por campo:
Campo
Type                      108
Amount                     20
Lead_Office                18
Lead_Source                15
Stage                      12
Amount_Currency             1
Delivery_Length_Months      1


In [35]:
# HTML do Relatório

cat_counts = errors_df['Categoria'].value_counts().to_dict()
cat_opp_counts = errors_df.groupby('Categoria')['Oportunidade'].nunique().to_dict()

categorias = {
    'Erro de grafia': {
        'icon': '&#9999;',
        'desc': 'Valores escritos de forma incorreta ou inconsistente em campos categóricos (Stage, Lead_Source, Lead_Office).',
        'impacto': 'Dados não seriam agrupados corretamente em análises e relatórios.',
        'correcao': 'Mapeamento para os valores padronizados definidos no case brief.'
    },
    'Divergência numérica': {
        'icon': '&#128290;',
        'desc': 'O campo Amount não corresponde à soma dos valores individuais de produto (Total_Product_Amount).',
        'impacto': 'Relatórios de receita apresentariam valores incorretos.',
        'correcao': 'Amount recalculado como soma de Total_Product_Amount (fonte de verdade conforme case brief).'
    },
    'Formato numérico': {
        'icon': '&#128221;',
        'desc': 'Valores numéricos com formatação inconsistente: prefixo R$, espaços extras, vírgula como separador de milhar.',
        'impacto': 'Cálculos automáticos falhariam ao interpretar esses valores.',
        'correcao': 'Remoção de prefixos e normalização de separadores.'
    },
    'Inconsistência de moeda': {
        'icon': '&#128177;',
        'desc': 'Uma oportunidade registrada em USD quando todo o restante do dataset usa BRL.',
        'impacto': 'Comparações de valor seriam distorcidas sem conversão cambial.',
        'correcao': 'Sinalizado para revisão manual.'
    },
    'Fora do escopo': {
        'icon': '&#128683;',
        'desc': 'Registros com tipos fora do escopo da análise: Flex/Renewal, Retainer, Passthrough ou sem tipo preenchido.',
        'impacto': 'Inclusão distorceria métricas de pipeline e conversão.',
        'correcao': 'Excluídos das análises da Parte 2.'
    }
}

def exemplos_html(cat, n=5):
    sub = errors_df[errors_df['Categoria'] == cat].head(n)
    rows = ''
    for _, e in sub.iterrows():
        rows += f'<tr><td>{e["Oportunidade"]}</td><td>{e["Campo"]}</td><td><code>{e["Valor original"]}</code></td><td><code>{e["Valor corrigido"]}</code></td></tr>\n'
    return rows

html = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Relatório de Erros</title>
<style>
  body {{ font-family: Arial, sans-serif; background: #0f1117; color: #e4e6ef; padding: 2rem; margin: 0; }}
  .container {{ max-width: 960px; margin: 0 auto; }}
  h1 {{ font-size: 2rem; color: #8187d4; margin-bottom: 0.25rem; }}
  .subtitle {{ color: #8b8fa3; margin-bottom: 2rem; }}
  .grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap: 1rem; margin-bottom: 2rem; }}
  .card {{ background: #1a1d27; border: 1px solid #2e3344; border-radius: 12px; padding: 1.25rem; text-align: center; }}
  .card .num {{ font-size: 2rem; font-weight: bold; color: #8187d4; }}
  .card .lbl {{ font-size: 0.8rem; color: #8b8fa3; }}
  .section {{ background: #1a1d27; border: 1px solid #2e3344; border-radius: 12px; padding: 1.5rem; margin-bottom: 1.5rem; }}
  .section h2 {{ font-size: 1.1rem; margin-bottom: 0.5rem; }}
  .badge {{ background: #242836; border: 1px solid #2e3344; border-radius: 20px; padding: 0.15rem 0.6rem; font-size: 0.75rem; color: #8187d4; }}
  .desc {{ color: #8b8fa3; font-size: 0.9rem; margin-bottom: 0.5rem; }}
  .impacto {{ color: #fbbf24; font-size: 0.85rem; margin-bottom: 0.5rem; padding-left: 1rem; border-left: 2px solid #fbbf24; }}
  .correcao {{ color: #4ade80; font-size: 0.85rem; margin-bottom: 1rem; padding-left: 1rem; border-left: 2px solid #4ade80; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; }}
  th {{ text-align: left; padding: 0.6rem; background: #242836; color: #8b8fa3; font-size: 0.75rem; text-transform: uppercase; }}
  td {{ padding: 0.5rem 0.6rem; border-top: 1px solid #2e3344; }}
  code {{ font-family: monospace; background: #242836; padding: 0.1rem 0.4rem; border-radius: 4px; font-size: 0.8rem; }}
  .footer {{ text-align: center; color: #8b8fa3; font-size: 0.8rem; margin-top: 2rem; padding-top: 1rem; border-top: 1px solid #2e3344; }}
</style>
</head>
<body>
<div class="container">
  <h1>Relatório de Erros - Auditoria de Dados</h1>
  <p class="subtitle">Case Técnico Operations - Monks</p>
  <div class="grid">
    <div class="card"><div class="num">{len(errors_log)}</div><div class="lbl">Problemas encontrados</div></div>
    <div class="card"><div class="num">{errors_df["Oportunidade"].nunique()}</div><div class="lbl">Oportunidades afetadas</div></div>
    <div class="card"><div class="num">5</div><div class="lbl">Categorias de erro</div></div>
    <div class="card"><div class="num">413</div><div class="lbl">Linhas auditadas</div></div>
  </div>
'''

for cat_name in ['Erro de grafia', 'Formato numérico', 'Divergência numérica', 'Inconsistência de moeda', 'Fora do escopo']:
    info = categorias[cat_name]
    cnt = cat_counts.get(cat_name, 0)
    opp_cnt = cat_opp_counts.get(cat_name, 0)
    exemplos = exemplos_html(cat_name)
    html += f'''
  <div class="section">
    <h2>{info["icon"]} {cat_name} <span class="badge">{cnt} ocorrências - {opp_cnt} oportunidades</span></h2>
    <p class="desc">{info["desc"]}</p>
    <p class="impacto">Impacto: {info["impacto"]}</p>
    <p class="correcao">Correção: {info["correcao"]}</p>
    <table>
      <thead><tr><th>Oportunidade</th><th>Campo</th><th>Valor original</th><th>Valor corrigido</th></tr></thead>
      <tbody>{exemplos}</tbody>
    </table>
  </div>
'''

html += '''
  <div class="section">
    <h2>&#128202; Observações sobre granularidade</h2>
    <p class="desc">A planilha está em nível de produto: uma mesma oportunidade pode aparecer em várias linhas (uma para cada produto do negócio). Existem linhas onde o mesmo Opportunity_ID + Product_Name aparece mais de uma vez, porém com valores de TPA diferentes — são itens distintos, não duplicatas.</p>
    <p class="correcao">Nenhuma linha foi removida. Para contagem de oportunidades, usou-se Opportunity_ID (261 total, 203 no escopo).</p>
  </div>
  <div class="footer">Gabriel Danius - Case Técnico Monks - Abril 2026</div>
</div>
</body>
</html>'''

with open('relatorio_erros.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('relatorio_erros.html gerado com sucesso!')
print(f'Total de problemas identificados: {len(errors_log)}')
print(f'Oportunidades afetadas: {errors_df["Oportunidade"].nunique()}')
print(f'Categorias de erro: {errors_df["Categoria"].nunique()}')

relatorio_erros.html gerado com sucesso!
Total de problemas identificados: 175
Oportunidades afetadas: 104
Categorias de erro: 5


Etapa 3 — Análise dos Dados (analise.html)

Usando a planilha já corrigida, irei gerar um relatório HTML com 10 gráficos e tabelas interativos.

In [38]:
valid_types = ['Project - Competitive', 'Project - Not Competitive', 'Change Order/Upsell']
df_scope = df[df['Type'].isin(valid_types)].copy()

df_scope['Close_Date_Dt'] = pd.to_datetime(df_scope['Close_Date'], format='%d/%m/%Y', errors='coerce')
df_scope['Created_Date_Dt'] = pd.to_datetime(df_scope['Created_Date'], format='%d/%m/%Y', errors='coerce')

df_cw = df_scope[df_scope['Stage'] == 'Closed Won'].copy()
df_open = df_scope[df_scope['Stage'] != 'Closed Won'].copy()

cw_deals = df_cw.groupby('Opportunity_ID').agg(
    Amount=('Amount_Parsed', 'first'),
    Close_Month=('Close_Date_Dt', lambda x: x.iloc[0].strftime('%Y-%m'))
).reset_index()
mom = cw_deals.groupby('Close_Month')['Amount'].sum().reset_index().sort_values('Close_Month')

cw_deals2 = df_cw.groupby('Opportunity_ID').agg(
    Amount=('Amount_Parsed', 'first'),
    LSC=('Lead_Source_Category', 'first')
).reset_index()
lsc_share = cw_deals2.groupby('LSC')['Amount'].sum()
lsc_pct = (lsc_share / lsc_share.sum() * 100).round(1)

open_wv = df_open[df_open['Amount_Parsed'].notna() & (df_open['Amount_Parsed'] > 0)]
top10_open = open_wv.groupby('Opportunity_ID').agg(
    Account=('Account_Name', 'first'),
    Products=('Product_Name', lambda x: ', '.join(x.dropna().unique())),
    Stage=('Stage', 'first'),
    Amount=('Amount_Parsed', 'first'),
    Close_Date=('Close_Date', 'first')
).reset_index().sort_values('Amount', ascending=False).head(10)

all_deals = df_scope.groupby('Opportunity_ID').agg(
    Stage=('Stage', 'first'),
    LSC=('Lead_Source_Category', 'first')
).reset_index()
wr_data = []
for lsc in all_deals['LSC'].dropna().unique():
    sub = all_deals[all_deals['LSC'] == lsc]
    total = len(sub)
    won = len(sub[sub['Stage'] == 'Closed Won'])
    wr_data.append({'LSC': lsc, 'Total': total, 'Won': won, 'Win_Rate': round(won/total*100, 1) if total else 0})
wr_df = pd.DataFrame(wr_data).sort_values('Win_Rate', ascending=False)

deals_by_type = df_scope.groupby('Opportunity_ID').agg(
    Type=('Type', 'first'),
    Amount=('Amount_Parsed', 'first')
).reset_index()
deals_wv = deals_by_type[deals_by_type['Amount'].notna() & (deals_by_type['Amount'] > 0)]
ticket_medio = deals_wv.groupby('Type')['Amount'].mean().round(0)

open_stage = df_open.groupby('Opportunity_ID').agg(
    Stage=('Stage', 'first'),
    Amount=('Amount_Parsed', 'first')
).reset_index()
open_stage_wv = open_stage[open_stage['Amount'].notna()]
pipeline_stage = open_stage_wv.groupby('Stage')['Amount'].sum().sort_values(ascending=False)

deals_mix = df_scope.groupby('Opportunity_ID').agg(
    Type=('Type', 'first'),
    Amount=('Amount_Parsed', 'first'),
    Close_Month=('Close_Date_Dt', lambda x: x.iloc[0].strftime('%Y-%m'))
).reset_index()
deals_mix['Biz_Type'] = deals_mix['Type'].apply(
    lambda x: 'Change Order/Upsell' if x == 'Change Order/Upsell' else 'New Business'
)
deals_mix_wv = deals_mix[deals_mix['Amount'].notna()]
mix_pivot = deals_mix_wv.pivot_table(
    index='Close_Month', columns='Biz_Type', values='Amount', aggfunc='sum', fill_value=0
)

cw_cycle = df_cw.groupby('Opportunity_ID').agg(
    Created=('Created_Date_Dt', 'first'),
    Closed=('Close_Date_Dt', 'first'),
    Office=('Lead_Office', 'first')
).reset_index()
cw_cycle['Days'] = (cw_cycle['Closed'] - cw_cycle['Created']).dt.days
cycle_office = cw_cycle.groupby('Office')['Days'].mean().round(0)

cw_clients = df_cw.groupby('Opportunity_ID').agg(
    Account=('Account_Name', 'first'),
    Amount=('Amount_Parsed', 'first')
).reset_index()
top_clients = cw_clients.groupby('Account')['Amount'].sum().sort_values(ascending=False).head(10)

today = pd.Timestamp('2026-04-22')
open_age = df_open.groupby('Opportunity_ID').agg(
    Created=('Created_Date_Dt', 'first')
).reset_index()
open_age['Age_Days'] = (today - open_age['Created']).dt.days

print(f'Dados no escopo: {len(df_scope)} linhas, {df_scope["Opportunity_ID"].nunique()} oportunidades')
print(f'Closed Won: {df_cw["Opportunity_ID"].nunique()} oportunidades, R$ {cw_deals["Amount"].sum():,.0f}')
print(f'Open Pipeline: {df_open["Opportunity_ID"].nunique()} oportunidades')
print(f'\nTodas as métricas calculadas!')

Dados no escopo: 305 linhas, 203 oportunidades
Closed Won: 67 oportunidades, R$ 27,186,886
Open Pipeline: 136 oportunidades

Todas as métricas calculadas!


Gerando o arquivo analise.html com os 10 gráficos

In [39]:
import json

mom_labels = ['Jan/26', 'Fev/26', 'Mar/26', 'Abr/26']
mom_values = mom['Amount'].tolist()

lsc_labels = lsc_pct.index.tolist()
lsc_values = lsc_pct.values.tolist()

top10_rows = ''
for i, (_, deal) in enumerate(top10_open.iterrows(), 1):
    prods = str(deal['Products'])
    if len(prods) > 80:
        prods = prods[:77] + '...'
    top10_rows += f'<tr><td>{i}</td><td>{deal["Account"]}</td><td>{prods}</td><td>{deal["Stage"]}</td><td class="num">R$ {deal["Amount"]:,.0f}</td><td class="num">{deal["Close_Date"]}</td></tr>\n'

html = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Análise de Dados - Pipeline Comercial</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  body {{ font-family: Arial, sans-serif; background: #0f1117; color: #e4e6ef; padding: 2rem; margin: 0; }}
  .container {{ max-width: 960px; margin: 0 auto; }}
  h1 {{ font-size: 2rem; color: #8187d4; margin-bottom: 0.25rem; }}
  .subtitle {{ color: #8b8fa3; margin-bottom: 2rem; }}
  .section {{ background: #1a1d27; border: 1px solid #2e3344; border-radius: 12px; padding: 1.5rem; margin-bottom: 1.5rem; }}
  .section h2 {{ font-size: 1.1rem; margin-bottom: 0.5rem; }}
  .obs {{ color: #8b8fa3; font-size: 0.85rem; margin-top: 0.75rem; padding-left: 1rem; border-left: 2px solid #6c72cb; font-style: italic; }}
  .chart-box {{ position: relative; max-height: 380px; margin: 1rem 0; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; margin-top: 0.75rem; }}
  th {{ text-align: left; padding: 0.6rem; background: #242836; color: #8b8fa3; font-size: 0.75rem; text-transform: uppercase; }}
  td {{ padding: 0.5rem 0.6rem; border-top: 1px solid #2e3344; }}
  .num {{ text-align: right; font-family: monospace; font-size: 0.8rem; }}
  .footer {{ text-align: center; color: #8b8fa3; font-size: 0.8rem; margin-top: 2rem; padding-top: 1rem; border-top: 1px solid #2e3344; }}
</style>
</head>
<body>
<div class="container">
  <h1>Análise de Dados - Pipeline Comercial</h1>
  <p class="subtitle">Case Técnico Operations - Monks - 305 linhas, 203 deals no escopo</p>

  <div class="section">
    <h2>1. Receita Closed Won - Mês a Mês (2026)</h2>
    <div class="chart-box"><canvas id="c1"></canvas></div>
    <p class="obs">Jan e Fev concentraram ~72% da receita fechada. Queda em Mar-Abr sugere sazonalidade ou pipeline em maturação.</p>
  </div>

  <div class="section">
    <h2>2. Participação % por Lead Source (Closed Won)</h2>
    <div class="chart-box" style="max-height:320px"><canvas id="c2"></canvas></div>
    <p class="obs">Customer Success domina com 65% da receita - maior parte vem de expansão em clientes existentes.</p>
  </div>

  <div class="section">
    <h2>3. Top 10 Oportunidades em Aberto</h2>
    <table>
      <thead><tr><th>#</th><th>Cliente</th><th>Produtos</th><th>Estágio</th><th>Valor</th><th>Fechamento</th></tr></thead>
      <tbody>{top10_rows}</tbody>
    </table>
    <p class="obs">As 10 maiores oportunidades somam ~R$ 32M. Jasper Security lidera com R$ 6.2M em Negotiation.</p>
  </div>
'''

print('Parte 1 do HTML gerada (gráficos 1 a 3)')

Parte 1 do HTML gerada (gráficos 1 a 3)


In [40]:
wr_labels = wr_df['LSC'].tolist()
wr_values = wr_df['Win_Rate'].tolist()
wr_totals = wr_df['Total'].tolist()

tm_labels = ticket_medio.index.tolist()
tm_values = ticket_medio.values.tolist()

ps_labels = pipeline_stage.index.tolist()
ps_values = pipeline_stage.values.tolist()

mix_months = sorted(mix_pivot.index.tolist())
mix_nb = [float(mix_pivot.loc[m].get('New Business', 0)) for m in mix_months]
mix_upsell = [float(mix_pivot.loc[m].get('Change Order/Upsell', 0)) for m in mix_months]
mix_display = ['Jan/26', 'Fev/26', 'Mar/26', 'Abr/26', 'Mai/26', 'Jun/26', 'Jul/26']

html += f'''
  <div class="section">
    <h2>4. Win Rate por Lead Source</h2>
    <div class="chart-box"><canvas id="c4"></canvas></div>
    <p class="obs">Customer Success tem o maior win rate (43.5%). Outbound alto (33.3%), mas com base pequena (3 deals).</p>
  </div>

  <div class="section">
    <h2>5. Ticket Médio por Tipo de Oportunidade</h2>
    <div class="chart-box"><canvas id="c5"></canvas></div>
    <p class="obs">Projetos competitivos: ticket 2.8x maior que não competitivos (R$ 963K vs R$ 347K).</p>
  </div>

  <div class="section">
    <h2>6. Pipeline em Aberto por Estágio</h2>
    <div class="chart-box"><canvas id="c6"></canvas></div>
    <p class="obs">Pitched + Negotiation = 67% do pipeline total em aberto.</p>
  </div>

  <div class="section">
    <h2>7. Mix New Business vs Upsell ao Longo do Tempo</h2>
    <div class="chart-box"><canvas id="c7"></canvas></div>
    <p class="obs">Maio concentra o maior volume (~R$ 36M), com New Business em 60%. Meses fechados: upsell predomina.</p>
  </div>
'''

print('Parte 2 do HTML gerada (gráficos 4 a 7)')

Parte 2 do HTML gerada (gráficos 4 a 7)


In [41]:
cy_labels = cycle_office.index.tolist()
cy_values = [float(v) for v in cycle_office.values]

tc_labels = top_clients.index.tolist()
tc_values = [float(v) for v in top_clients.values]

ages = open_age['Age_Days'].tolist()
bins_edges = [0, 30, 60, 90, 120, 180, 365]
bin_labels_hist = ['0-30d', '31-60d', '61-90d', '91-120d', '121-180d', '180d+']
hist_counts = [sum(1 for a in ages if bins_edges[i] <= a < bins_edges[i+1]) for i in range(len(bins_edges)-1)]

html += f'''
  <div class="section">
    <h2>8. Ciclo de Venda Médio (Dias) por Lead Office</h2>
    <div class="chart-box" style="max-height:280px"><canvas id="c8"></canvas></div>
    <p class="obs">São Paulo tem o ciclo mais longo (91 dias). São Carlos é o mais rápido (67 dias).</p>
  </div>

  <div class="section">
    <h2>9. Top 10 Clientes - Closed Won (YTD 2026)</h2>
    <div class="chart-box"><canvas id="c9"></canvas></div>
    <p class="obs">Top 3 clientes concentram ~48% da receita. Alta concentração = risco de dependência.</p>
  </div>

  <div class="section">
    <h2>10. Idade do Pipeline Aberto</h2>
    <div class="chart-box"><canvas id="c10"></canvas></div>
    <p class="obs">Maioria com menos de 60 dias. Oportunidades 180d+ merecem revisão - possíveis deals estagnados.</p>
  </div>

  <div class="footer">Gabriel Danius - Case Técnico Monks - Abril 2026</div>
</div>
'''

print('Parte 3 do HTML gerada (gráficos 8 a 10)')

Parte 3 do HTML gerada (gráficos 8 a 10)


In [44]:
chart_js_url = "https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"

import urllib.request
chart_js_code = urllib.request.urlopen(chart_js_url).read().decode('utf-8')

html += f'''
<script>
{chart_js_code}
</script>
<script>
Chart.defaults.color = "#8b8fa3";
Chart.defaults.borderColor = "#2e3344";
Chart.defaults.font.family = "Arial, sans-serif";
var P = ["#6c72cb","#60a5fa","#4ade80","#fbbf24","#f87171","#a78bfa","#34d399","#fb923c"];

new Chart(document.getElementById("c1"),{{type:"bar",data:{{labels:{json.dumps(mom_labels)},datasets:[{{data:{json.dumps(mom_values)},backgroundColor:"#6c72cb",borderRadius:6}}]}},options:{{responsive:true,plugins:{{legend:{{display:false}},tooltip:{{callbacks:{{label:function(c){{return"R$ "+c.parsed.y.toLocaleString("pt-BR")}}}}}}}},scales:{{y:{{ticks:{{callback:function(v){{return"R$ "+(v/1e6).toFixed(1)+"M"}}}}}}}}}}}});

new Chart(document.getElementById("c2"),{{type:"doughnut",data:{{labels:{json.dumps(lsc_labels)},datasets:[{{data:{json.dumps(lsc_values)},backgroundColor:P,borderWidth:0}}]}},options:{{responsive:true,plugins:{{legend:{{position:"right"}},tooltip:{{callbacks:{{label:function(c){{return c.label+": "+c.parsed+"%"}}}}}}}}}}}});

new Chart(document.getElementById("c4"),{{type:"bar",data:{{labels:{json.dumps(wr_labels)},datasets:[{{data:{json.dumps(wr_values)},backgroundColor:"#4ade80",borderRadius:6}}]}},options:{{responsive:true,indexAxis:"y",plugins:{{legend:{{display:false}}}},scales:{{x:{{max:50,ticks:{{callback:function(v){{return v+"%"}}}}}}}}}}}});

new Chart(document.getElementById("c5"),{{type:"bar",data:{{labels:{json.dumps(tm_labels)},datasets:[{{data:{json.dumps(tm_values)},backgroundColor:["#60a5fa","#6c72cb","#fbbf24"],borderRadius:6}}]}},options:{{responsive:true,plugins:{{legend:{{display:false}}}},scales:{{y:{{ticks:{{callback:function(v){{return"R$ "+(v/1e3).toFixed(0)+"K"}}}}}}}}}}}});

new Chart(document.getElementById("c6"),{{type:"bar",data:{{labels:{json.dumps(ps_labels)},datasets:[{{data:{json.dumps(ps_values)},backgroundColor:"#60a5fa",borderRadius:6}}]}},options:{{responsive:true,indexAxis:"y",plugins:{{legend:{{display:false}}}},scales:{{x:{{ticks:{{callback:function(v){{return"R$ "+(v/1e6).toFixed(0)+"M"}}}}}}}}}}}});

new Chart(document.getElementById("c7"),{{type:"bar",data:{{labels:{json.dumps(mix_display)},datasets:[{{label:"New Business",data:{json.dumps(mix_nb)},backgroundColor:"#6c72cb",borderRadius:4}},{{label:"Change Order/Upsell",data:{json.dumps(mix_upsell)},backgroundColor:"#60a5fa",borderRadius:4}}]}},options:{{responsive:true,scales:{{x:{{stacked:true}},y:{{stacked:true,ticks:{{callback:function(v){{return"R$ "+(v/1e6).toFixed(0)+"M"}}}}}}}}}}}});

new Chart(document.getElementById("c8"),{{type:"bar",data:{{labels:{json.dumps(cy_labels)},datasets:[{{data:{json.dumps(cy_values)},backgroundColor:["#6c72cb","#60a5fa","#4ade80"],borderRadius:6}}]}},options:{{responsive:true,plugins:{{legend:{{display:false}}}},scales:{{y:{{beginAtZero:true,max:120}}}}}}}});

new Chart(document.getElementById("c9"),{{type:"bar",data:{{labels:{json.dumps(tc_labels)},datasets:[{{data:{json.dumps(tc_values)},backgroundColor:"#a78bfa",borderRadius:6}}]}},options:{{responsive:true,indexAxis:"y",plugins:{{legend:{{display:false}}}},scales:{{x:{{ticks:{{callback:function(v){{return"R$ "+(v/1e6).toFixed(1)+"M"}}}}}}}}}}}});

new Chart(document.getElementById("c10"),{{type:"bar",data:{{labels:{json.dumps(bin_labels_hist)},datasets:[{{data:{json.dumps(hist_counts)},backgroundColor:"#fbbf24",borderRadius:6}}]}},options:{{responsive:true,plugins:{{legend:{{display:false}}}},scales:{{y:{{beginAtZero:true,title:{{display:true,text:"Nº de Oportunidades"}}}}}}}}}});
</script>
</body>
</html>'''

with open('analise.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'analise.html gerado com sucesso!')
print(f'10 gráficos/tabelas criados')

analise.html gerado com sucesso!
10 gráficos/tabelas criados


Os gráficos estão no arquivo analise.html, que pode ser visualizado abrindo no navegador.